# Project 2 — Digital Advertising Budget Optimizer
**Student:** Javeria | **Course:** Big Data Analytics

**Sawal (Industry problem):** Company Facebook, Instagram, Google, TikTok aur YouTube par paisa kharch karti hai. Management poochti hai: **"Agle Rs. 1 million kahan lagayein?"**

**Is notebook mein hum ye karenge (7 steps):**
1. Setup
2. Simulated ads data generate karna (10K se 10 million records tak)
3. KPIs calculate karna: CTR, CPC, CPA, Conversion Rate, ROAS, Revenue
4. Pandas vs MapReduce vs Spark ka comparison (speed test)
5. Campaign performance
6. Marketing Budget Recommendation Engine
7. Final summary

**Ahem:** Har cell ko upar se neeche **ek ek karke** run karein (Shift + Enter).

## Step 1: Setup
Pehle libraries install aur import karte hain. (Pehli baar 1-2 minute lag sakte hain.)

In [ ]:
!pip install -q pyspark pyarrow

In [ ]:
import os, time, gc, warnings
from collections import defaultdict
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

OUT = "project2_ad_optimizer"           # saara output isi folder mein jayega
os.makedirs(f"{OUT}/data", exist_ok=True)
os.makedirs(f"{OUT}/results", exist_ok=True)

PLATFORMS = ["Facebook", "Instagram", "Google", "TikTok", "YouTube"]
SUM_COLS = ["impressions", "clicks", "spend", "conversions", "revenue"]
print("Setup ho gaya")

## Step 2: Data Simulation (nakli lekin realistic data)
Hamare paas asli company ka data nahi hai, is liye hum **simulation** se data banate hain.
Har platform ka apna "asli behaviour" hai (CTR, CPC, conversion rate, order value), aur data usi ke gird random bana hai.
Phir hamari analysis ko ye behaviour **wapis dhoondna** hai.

Columns: `date, campaign, platform, impressions, clicks, spend, conversions, revenue, age, gender, city, device`
(Paisa Pakistani Rupees (Rs.) mein hai.)

In [ ]:
# Har platform ka "hidden truth" (Rs. mein)
TRUE = pd.DataFrame({
    "platform":      PLATFORMS,
    "ctr":           [0.010, 0.013, 0.030, 0.018, 0.006],   # click-through rate
    "cpc":           [30,    38,    45,    22,    18   ],   # cost per click (Rs.)
    "cvr":           [0.030, 0.028, 0.055, 0.016, 0.018],   # conversion rate
    "aov":           [4500,  4800,  5200,  3800,  4200 ],   # average order value (Rs.)
    "traffic_share": [0.30,  0.21,  0.03,  0.21,  0.25 ],   # is platform ke records ka hissa
}).set_index("platform")

AGES    = ["18-24", "25-34", "35-44", "45+"]; AGE_P = [.30, .35, .20, .15]
AGE_CTR = np.array([1.2, 1.1, 0.9, 0.7]);     AGE_CVR = np.array([0.8, 1.1, 1.2, 1.1])
GENDERS = ["Male", "Female"]
CITIES  = ["Lahore", "Karachi", "Islamabad", "Faisalabad", "Peshawar"]
CITY_P  = [.28, .32, .14, .14, .12];          CITY_CVR = np.array([1.05, 1.0, 1.2, 0.9, 0.85])
DEVICES = ["Mobile", "Desktop", "Tablet"];    DEVICE_P = [.70, .22, .08]
DEVICE_CVR = np.array([0.9, 1.3, 1.0])
CAMP_PER_PLATFORM = 6
CAMPAIGNS = [f"{p}_Campaign_{k+1}" for p in PLATFORMS for k in range(CAMP_PER_PLATFORM)]

def generate_data(n, seed=42):
    rng = np.random.default_rng(seed)
    p      = rng.choice(len(PLATFORMS), size=n, p=TRUE["traffic_share"].values)
    camp   = p * CAMP_PER_PLATFORM + rng.integers(0, CAMP_PER_PLATFORM, size=n)
    age    = rng.choice(len(AGES),    size=n, p=AGE_P)
    gender = rng.choice(len(GENDERS), size=n)
    city   = rng.choice(len(CITIES),  size=n, p=CITY_P)
    device = rng.choice(len(DEVICES), size=n, p=DEVICE_P)

    impressions = rng.integers(2000, 20000, size=n).astype(np.int32)
    ctr    = TRUE["ctr"].values[p] * AGE_CTR[age]
    clicks = rng.binomial(impressions, np.clip(ctr, 0, 1)).astype(np.int32)
    cpc    = rng.gamma(5.0, TRUE["cpc"].values[p] / 5.0)
    spend  = clicks * cpc
    cvr    = TRUE["cvr"].values[p] * AGE_CVR[age] * CITY_CVR[city] * DEVICE_CVR[device]
    conversions = rng.binomial(clicks, np.clip(cvr, 0, 1)).astype(np.int32)
    revenue = conversions * TRUE["aov"].values[p] * rng.lognormal(0, 0.25, size=n)
    date = np.datetime64("2025-01-01") + rng.integers(0, 365, size=n).astype("timedelta64[D]")

    return pd.DataFrame({
        "date": date,
        "campaign": pd.Categorical.from_codes(camp, CAMPAIGNS),
        "platform": pd.Categorical.from_codes(p, PLATFORMS),
        "impressions": impressions, "clicks": clicks,
        "spend": spend, "conversions": conversions, "revenue": revenue,
        "age": pd.Categorical.from_codes(age, AGES),
        "gender": pd.Categorical.from_codes(gender, GENDERS),
        "city": pd.Categorical.from_codes(city, CITIES),
        "device": pd.Categorical.from_codes(device, DEVICES),
    })

sample = generate_data(10_000)
sample.head(10)

## Step 3: KPIs
Marketing ke 5 bunyadi numbers:

| KPI | Formula | Matlab |
|---|---|---|
| CTR | clicks / impressions | Ad dekhne walon mein se kitnon ne click kiya |
| CPC | spend / clicks | Ek click ki qeemat |
| CPA | spend / conversions | Ek customer lene ki qeemat |
| Conversion Rate | conversions / clicks | Click karne walon mein se kitnon ne khareeda |
| ROAS | revenue / spend | Har Rs. 1 kharch par kitne Rs. wapis aaye |

In [ ]:
def add_kpis(t):
    t = t.copy()
    t["CTR_%"]  = t["clicks"] / t["impressions"] * 100
    t["CPC"]    = t["spend"] / t["clicks"]
    t["CPA"]    = t["spend"] / t["conversions"]
    t["CVR_%"]  = t["conversions"] / t["clicks"] * 100
    t["ROAS"]   = t["revenue"] / t["spend"]
    t["profit"] = t["revenue"] - t["spend"]
    return t

by_platform = sample.groupby("platform", observed=True)[SUM_COLS].sum()
add_kpis(by_platform)

## Step 4: Big Data Simulation — Pandas vs MapReduce vs Spark
Hum data ko **10,000 → 100,000 → 1 million → 10 million** records tak le jayenge.
Teeno tools se **same kaam** karwayenge: *"har platform ka total impressions, clicks, spend, conversions, revenue nikalo."*
Phir dekhenge kaun kitni der leta hai.

- **Pandas:** ek machine, ek hi memory mein, `groupby().sum()`
- **MapReduce:** *Map* (har record ko `(platform, values)` bana do) → *Shuffle* (same platform ko ek jagah jama karo) → *Reduce* (jama kiye hue ko jod do). Yahan hum ye pure Python mein khud likh rahe hain aur multiple workers se chalate hain.
- **Spark:** professional big data engine, local mode mein.

**Note:** Teeno ka time **file parhne + hisaab karne** dono ka hai (data Parquet file mein save hai).

In [ ]:
# ---------- 1) PANDAS ----------
def pandas_job(path):
    df = pd.read_parquet(path, columns=["platform"] + SUM_COLS)
    return df.groupby("platform", observed=True)[SUM_COLS].sum()

# ---------- 2) MAPREDUCE (pure Python) ----------
_DATA = {}   # workers (fork) isi ko share karte hain

def mapper(bounds):
    start, end = bounds
    keys = _DATA["platform"][start:end].tolist()
    cols = [_DATA[c][start:end].tolist() for c in SUM_COLS]
    local = {}                                   # combiner: har worker apna chhota hisaab rakhta hai
    for k, imp, clk, sp, cv, rev in zip(keys, *cols):   # MAP: har record -> (key, values)
        acc = local.get(k)
        if acc is None:
            local[k] = [imp, clk, sp, cv, rev]
        else:
            acc[0] += imp; acc[1] += clk; acc[2] += sp; acc[3] += cv; acc[4] += rev
    return local

def reducer(key, list_of_values):                # REDUCE: ek key ki saari values jodo
    return [sum(v[i] for v in list_of_values) for i in range(5)]

def mapreduce_job(path, n_workers=2, n_chunks=8):
    df = pd.read_parquet(path, columns=["platform"] + SUM_COLS)
    _DATA["platform"] = df["platform"].cat.codes.to_numpy()
    for c in SUM_COLS:
        _DATA[c] = df[c].to_numpy()
    n = len(df)
    edges = np.linspace(0, n, n_chunks + 1).astype(int)
    bounds = list(zip(edges[:-1], edges[1:]))
    with mp.get_context("fork").Pool(n_workers) as pool:
        partials = pool.map(mapper, bounds)      # MAP phase (parallel)
    shuffle = defaultdict(list)                  # SHUFFLE phase: key ke hisaab se jama karo
    for part in partials:
        for k, v in part.items():
            shuffle[k].append(v)
    result = {PLATFORMS[k]: reducer(k, vals) for k, vals in shuffle.items()}   # REDUCE phase
    return pd.DataFrame.from_dict(result, orient="index", columns=SUM_COLS)

# ---------- 3) SPARK ----------
try:
    from pyspark.sql import SparkSession, functions as F
    spark = (SparkSession.builder.master("local[*]").appName("AdBudgetOptimizer")
             .config("spark.driver.memory", "4g")
             .config("spark.ui.showConsoleProgress", "false").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    spark.range(1000).count()                    # warm-up (JVM start ka time alag rakhne ke liye)
    HAS_SPARK = True
except Exception as e:
    HAS_SPARK = False
    print("Spark start nahi hua:", e)

def spark_job(path):
    sdf = spark.read.parquet(path)
    rows = sdf.groupBy("platform").agg(*[F.sum(c).alias(c) for c in SUM_COLS]).collect()
    return pd.DataFrame([r.asDict() for r in rows]).set_index("platform")

def timed(fn, path):
    t0 = time.perf_counter()
    res = fn(path)
    return res, time.perf_counter() - t0

def same_result(a, b):
    a = a.reindex(PLATFORMS)[SUM_COLS].astype(float); b = b.reindex(PLATFORMS)[SUM_COLS].astype(float)
    return np.allclose(a.values, b.values, rtol=1e-6)

print("Teeno jobs ready. Spark available:", HAS_SPARK)

In [ ]:
SIZES = [10_000, 100_000, 1_000_000, 10_000_000]
bench_rows = []
hist_df = None            # 1 million wala data aage recommendation ke liye rakhenge

for n in SIZES:
    print(f"\n=== {n:,} records ===")
    t0 = time.perf_counter()
    df = generate_data(n)
    path = f"{OUT}/data/ads_{n}.parquet"
    df.to_parquet(path, index=False, coerce_timestamps="ms", allow_truncated_timestamps=True)
    print(f"data bana aur save hua: {time.perf_counter()-t0:.1f}s | file size: {os.path.getsize(path)/1e6:.1f} MB")
    if n == 1_000_000:
        hist_df = df
    else:
        del df
    gc.collect()

    r_pd, t_pd = timed(pandas_job, path)
    r_mr, t_mr = timed(mapreduce_job, path)
    if HAS_SPARK:
        r_sp, t_sp = timed(spark_job, path)
    else:
        r_sp, t_sp = None, np.nan

    ok = same_result(r_pd, r_mr) and (r_sp is None or same_result(r_pd, r_sp))
    print(f"Pandas: {t_pd:.2f}s | MapReduce: {t_mr:.2f}s | Spark: {t_sp:.2f}s | teeno ka jawab same: {ok}")
    bench_rows.append({"records": n, "pandas_sec": t_pd, "mapreduce_sec": t_mr, "spark_sec": t_sp, "results_match": ok})

bench = pd.DataFrame(bench_rows)
bench.to_csv(f"{OUT}/results/benchmark.csv", index=False)
bench

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bench["records"], bench["pandas_sec"],    "o-", label="Pandas")
ax.plot(bench["records"], bench["mapreduce_sec"], "s-", label="MapReduce (Python)")
if HAS_SPARK:
    ax.plot(bench["records"], bench["spark_sec"], "^-", label="Spark")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Records (log scale)"); ax.set_ylabel("Time in seconds (log scale)")
ax.set_title("Pandas vs MapReduce vs Spark"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/results/benchmark.png", dpi=150); plt.show()

**Natija kaise samjhein (apni run ke numbers dekh kar):**
- Chhote data (10K) par Spark aksar **sab se slow** hota hai, kyunke uska startup/overhead hota hai. Pandas sab se tez.
- Data barhne par Pandas ka time barhta hai aur memory bhi zyada lagti hai.
- Pure Python MapReduce record-by-record (Python loop) chalta hai, is liye wo aksar Pandas se slow hota hai.
- Spark ka asli faida tab hai jab data ek machine ki memory se bara ho ya **cluster** (kai machines) par chalaya jaye. Colab mein sirf 2 cores hain, is liye Spark ki poori taqat yahan nazar nahi aati.

## Step 5: Campaign Performance
Ab 1 million records wale data se dekhte hain kaun si campaign sab se achi hai.

In [ ]:
plat_kpi = add_kpis(hist_df.groupby("platform", observed=True)[SUM_COLS].sum())
print("Platform-wise performance:")
display(plat_kpi.sort_values("ROAS", ascending=False))

camp_kpi = add_kpis(hist_df.groupby("campaign", observed=True)[SUM_COLS].sum()).sort_values("ROAS", ascending=False)
print("Top 5 campaigns (ROAS ke hisaab se):")
display(camp_kpi.head(5)[["spend", "revenue", "CTR_%", "CPA", "CVR_%", "ROAS"]])
print("Sab se kamzor 5 campaigns:")
display(camp_kpi.tail(5)[["spend", "revenue", "CTR_%", "CPA", "CVR_%", "ROAS"]])
plat_kpi.to_csv(f"{OUT}/results/platform_kpis.csv")
camp_kpi.to_csv(f"{OUT}/results/campaign_kpis.csv")

## Step 6: Marketing Budget Recommendation Engine
**Idea:** jis platform ka **ROAS** zyada, usko zyada budget. Lekin sara paisa ek hi platform par nahi lagate (risk), is liye:
- ROAS ko power `alpha = 1.5` se barhate hain (achay platform ko thora extra weight milta hai)
- Har platform ka share **minimum 5%** aur **maximum 40%** rakha hai (diversification)

In [ ]:
def recommend_budget(kpi, total_budget=1_000_000, alpha=1.5, min_share=0.05, max_share=0.40):
    w = kpi["ROAS"] ** alpha
    fixed = {}
    while True:
        free = [p for p in w.index if p not in fixed]
        remaining = 1 - sum(fixed.values())
        share = {p: w[p] / sum(w[q] for q in free) * remaining for p in free}
        bad = [p for p in free if share[p] < min_share or share[p] > max_share]
        if not bad:
            break
        for p in bad:
            fixed[p] = min_share if share[p] < min_share else max_share
    share.update(fixed)
    rec = kpi[["spend", "revenue", "ROAS", "CPA", "CVR_%"]].copy()
    rec["current_share_%"] = rec["spend"] / rec["spend"].sum() * 100
    rec["recommended_share_%"] = pd.Series(share) * 100
    rec["recommended_budget_Rs"] = pd.Series(share) * total_budget
    rec["expected_revenue_Rs"] = rec["recommended_budget_Rs"] * rec["ROAS"]
    return rec.sort_values("recommended_share_%", ascending=False)

BUDGET = 1_000_000
rec = recommend_budget(plat_kpi, BUDGET)
display(rec[["ROAS", "current_share_%", "recommended_share_%", "recommended_budget_Rs", "expected_revenue_Rs"]])

print("Marketing Budget Recommendation (agle Rs. 1 million ke liye):")
for p, r in rec.iterrows():
    print(f"  {p:<10}: {r['recommended_share_%']:.0f}%  ->  Rs. {r['recommended_budget_Rs']:,.0f}")

equal_rev   = (plat_kpi["ROAS"] * BUDGET / len(plat_kpi)).sum()
current_rev = (plat_kpi["ROAS"] * rec["current_share_%"] / 100 * BUDGET).sum()
new_rev     = rec["expected_revenue_Rs"].sum()
print(f"\nExpected revenue  | barabar taqseem (20% each): Rs. {equal_rev:,.0f}")
print(f"                  | purani taqseem (pichhle kharch jaisi): Rs. {current_rev:,.0f}")
print(f"                  | nayi recommendation: Rs. {new_rev:,.0f}  ({(new_rev/current_rev-1)*100:+.1f}% purani se behtar)")
rec.to_csv(f"{OUT}/results/budget_recommendation.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = plat_kpi["ROAS"].sort_values(ascending=False)
axes[0].bar(order.index, order.values, color="#2E75B6")
axes[0].set_title("ROAS by Platform (Rs. 1 kharch par kitna wapis)"); axes[0].set_ylabel("ROAS")
for i, v in enumerate(order.values): axes[0].text(i, v + 0.05, f"{v:.2f}", ha="center")

x = np.arange(len(rec)); wd = 0.38
axes[1].bar(x - wd/2, rec["current_share_%"], wd, label="Current", color="#BFBFBF")
axes[1].bar(x + wd/2, rec["recommended_share_%"], wd, label="Recommended", color="#1F3864")
axes[1].set_xticks(x); axes[1].set_xticklabels(rec.index)
axes[1].set_title("Budget Share: Current vs Recommended (%)"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/results/budget_recommendation.png", dpi=150); plt.show()

### Bonus: har platform ka sab se acha segment
Kis age group, device aur city par har platform ka ROAS sab se zyada hai? (Ye marketing team ko targeting mein madad deta hai.)

In [ ]:
def best_segment(col):
    g = add_kpis(hist_df.groupby(["platform", col], observed=True)[SUM_COLS].sum())
    return g["ROAS"].groupby(level="platform").idxmax().map(lambda t: t[1])

segments = pd.DataFrame({"best_age": best_segment("age"),
                         "best_device": best_segment("device"),
                         "best_city": best_segment("city")})
segments.to_csv(f"{OUT}/results/best_segments.csv")
segments

## Step 7: Final Summary

In [ ]:
top = rec.index[0]
print("PROJECT 2 — SUMMARY")
print("-" * 50)
print(f"1. Data sizes tested: {', '.join(f'{n:,}' for n in SIZES)} records")
print(f"2. Sab se behtar platform (ROAS): {plat_kpi['ROAS'].idxmax()}  ({plat_kpi['ROAS'].max():.2f})")
print(f"3. Sab se kamzor platform (ROAS): {plat_kpi['ROAS'].idxmin()}  ({plat_kpi['ROAS'].min():.2f})")
print(f"4. Recommendation: sab se zyada budget {top} ko ({rec.iloc[0]['recommended_share_%']:.0f}%)")
print(f"5. Nayi taqseem se expected revenue purani se {(new_rev/current_rev-1)*100:+.1f}% behtar")
print("\nSaari files yahan save hain:", OUT + "/results/")
os.listdir(f"{OUT}/results")